In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-getting-started/sample_submission.csv
/kaggle/input/nlp-getting-started/train.csv
/kaggle/input/nlp-getting-started/test.csv


In [2]:
!pip install datasets evaluate transformers[sentencepiece] nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.0 MB/s eta 0:00:00


In [3]:
import nltk
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

In [4]:
import pandas as pd

train_tweets=pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_tweets=pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [5]:
train_tweets.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [6]:
train_tweets.value_counts('target')

target
0    4342
1    3271
Name: count, dtype: int64

In [7]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
stop_words = set(stopwords.words('english'))

def clean_tweet(tweet):
    tweet = tweet.lower()  # Convert to lowercase
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet)  # Remove URLs
    tweet = re.sub(r'[^A-Za-z\s]', '', tweet)  # Remove punctuation and special characters
    tweet = ' '.join([word for word in tweet.split() if word not in stop_words])  # Remove stopwords
    return tweet

train_tweets['text'] = train_tweets['text'].apply(clean_tweet)
test_tweets['text'] = test_tweets['text'].apply(clean_tweet)

In [9]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_tweets['text'], train_tweets['target'], test_size=0.2, random_state=42
)

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(train_texts)
X_val_vec = vectorizer.transform(val_texts)

In [11]:
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, train_labels)
y_pred_nb = nb_model.predict(X_val_vec)

# Evaluate Naive Bayes
print(f"Naive Bayes Accuracy: {accuracy_score(val_labels, y_pred_nb)}")
print(confusion_matrix(val_labels, y_pred_nb))
print(classification_report(val_labels, y_pred_nb))

Naive Bayes Accuracy: 0.8036769533814839
[[782  92]
 [207 442]]
              precision    recall  f1-score   support

           0       0.79      0.89      0.84       874
           1       0.83      0.68      0.75       649

    accuracy                           0.80      1523
   macro avg       0.81      0.79      0.79      1523
weighted avg       0.81      0.80      0.80      1523



In [12]:
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_vec, train_labels)
y_pred_svm = svm_model.predict(X_val_vec)

# Evaluate SVM
print(f"SVM Accuracy: {accuracy_score(val_labels, y_pred_svm)}")
print(confusion_matrix(val_labels, y_pred_svm))
print(classification_report(val_labels, y_pred_svm))

SVM Accuracy: 0.7964543663821405
[[760 114]
 [196 453]]
              precision    recall  f1-score   support

           0       0.79      0.87      0.83       874
           1       0.80      0.70      0.75       649

    accuracy                           0.80      1523
   macro avg       0.80      0.78      0.79      1523
weighted avg       0.80      0.80      0.79      1523



In [13]:
sample_submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
sample_submission.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [14]:
X_test_vec = vectorizer.transform(test_tweets['text'])
test_preds = nb_model.predict(X_test_vec)
sample_submission["target"] = test_preds
sample_submission.to_csv('submission.csv', index=False)


print("Corrected submission file saved as 'submission.csv'")

Corrected submission file saved as 'submission.csv'


In [15]:
sample_submission.head()  

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
